Last time we learned how to reduce the number of messages in a state.

We removed messages from the state by adding them.

It's a quirky concept, kind of like how cells multiply by dividing.

We'll now implement yet another, more efficient approach for long running conversations.

Instead of storing previous interactions verbatim, we'll summarize them.

The game plan is as follows.

First, we'll create a new graph state inheriting from the messages state class.

It will have an additional parameter summary where we'll store the conversations running.

Summary.

Second, we'll modify the chatbot node to feed the user's question and the summary.

Finally, we'll substitute the trim messages node with one that creates a new summary or extends an

existing one.

Once done, it deletes all messages in the state.

So let's begin.



In [7]:
import config
from langgraph.graph import START, END, StateGraph, add_messages,MessagesState
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, RemoveMessage
from typing import Literal

In [8]:
class State(MessagesState):
    summary: str

In [9]:
chat = ChatOpenAI(
    model="gpt-3.5-turbo",
    seed= 365,
    temperature=0,
    max_completion_tokens=50,
    openai_api_key = config.api_key
)

In [10]:
def ask_question (state: State) -> State:
    print(f"\n--------> ENTERING ask_question")
    
    question = "What is your question ?"
    print(question)
    return State(messages = [AIMessage(question), HumanMessage(input())])

In [11]:
def chatbot (state: State) -> State:
    print(f"\n--------> ENTERING CHATBOT")
    for i in state["messages"]:
        i.pretty_print()

    system_messages = f'''
    Here's a quick summary of what's been discussed so far: {state.get("summary", " ")}
    
    keep this in mind as you answer the next question. 
    '''
    response = chat.invoke( [SystemMessage(system_messages)]+state["messages"])
    response.pretty_print()

    return State(messages = [response])

In [12]:
def ask_another_question (state: State) -> State:
    print(f"\n--------> ENTERING ask_another_question")
    
    question = "Would you like to ask another question (YES/NO) ?"
    print(question)
    return State(messages = [AIMessage(question), HumanMessage(input())])

In [13]:
def summarize_and_delete_msg (state:State) -> State:
    print(f"\n--------> ENTERING trim_messages:")
    
    new_conversation= ""
    for i in state["messages"]:
        new_conversation += f"{i.type}: {i.content}\n\n"


    summary_instructions =f'''
    Update the ongoing summary by incorporating the new lines of conversation below.
    Build upon the previous summary rather than repeating it so that the result reflects the most recent
    context and developments.

    Previous_summary:
    {state.get("summary", "")}

    New_Conversation:
    {new_conversation}
    '''
    print(summary_instructions)
    summary = chat.invoke([HumanMessage(summary_instructions)])
    remove_messages = [RemoveMessage(id = i.id) for i in state["messages"][:]] #Modify the remove messages variable so all messages in the state are deleted.


    return State(messages = remove_messages, summary= summary.content)

In [14]:
#We expect to append each new message to the list, so we shouldn't be checking the first message, but the last one instead.
def routing_function(state: State) -> Literal["summarize_and_delete_msg", END]:
    if state["messages"][-1].content == "yes":
        return "summarize_and_delete_msg"
    else:
        return END

In [15]:
graph = StateGraph(State) ## here pass the schema i.e state class defined above

In [16]:
graph.add_node("ask_question", ask_question)

graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_node("summarize_and_delete_msg", summarize_and_delete_msg)

graph.add_edge(START , "ask_question")
graph.add_edge("ask_question" , "chatbot")
graph.add_edge("chatbot" , "ask_another_question")

graph.add_conditional_edges(source = "ask_another_question",
                           path= routing_function,
                           )
graph.add_edge("summarize_and_delete_msg" , "ask_question")


In [17]:
grpah_compiled = graph.compile()

In [18]:
grpah_compiled

ValueError: Failed to reach https://mermaid.ink API while trying to render your graph. Status code: 503.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [20]:
grpah_compiled.invoke(State(message=[] ))


--------> ENTERING ask_question
What is your question ?


 who is gandhi



--------> ENTERING CHATBOT
================================== Ai Message ==================================

What is your question ?
================================ Human Message =================================

who is gandhi
================================== Ai Message ==================================

Gandhi, also known as Mahatma Gandhi, was a prominent leader in the Indian independence movement against British colonial rule. He was known for his nonviolent resistance and civil disobedience tactics, which inspired movements for civil rights and freedom across the

--------> ENTERING ask_another_question
Would you like to ask another question (YES/NO) ?


 yes



--------> ENTERING trim_messages:

    Update the ongoing summary by incorporating the new lines of conversation below.
    Build upon the previous summary rather than repeating it so that the result reflects the most recent
    context and developments.

    Previous_summary:
    

    New_Conversation:
    ai: What is your question ?

human: who is gandhi

ai: Gandhi, also known as Mahatma Gandhi, was a prominent leader in the Indian independence movement against British colonial rule. He was known for his nonviolent resistance and civil disobedience tactics, which inspired movements for civil rights and freedom across the

ai: Would you like to ask another question (YES/NO) ?

human: yes


    

--------> ENTERING ask_question
What is your question ?


...


--------> ENTERING trim_messages:

    Update the ongoing summary by incorporating the new lines of conversation below.
    Build upon the previous summary rather than repeating it so that the result reflects the most recent
    context and developments.

    Previous_summary:
    Summary:
The AI provided information about Mahatma Gandhi, a prominent leader in the Indian independence movement known for his nonviolent resistance. The conversation ended with the AI asking if the human would like to ask another question, to which the human responded "

    New_Conversation:
    ai: What is your question ?

human: what year was he born

ai: Mahatma Gandhi was born on October 2, 1869.

ai: Would you like to ask another question (YES/NO) ?

human: yes


    

--------> ENTERING ask_question
What is your question ?


 what year he died?



--------> ENTERING CHATBOT
================================== Ai Message ==================================

What is your question ?
================================ Human Message =================================

what year he died?
================================== Ai Message ==================================

Mahatma Gandhi passed away on January 30, 1948.

--------> ENTERING ask_another_question
Would you like to ask another question (YES/NO) ?


 no


{'messages': [AIMessage(content='What is your question ?', additional_kwargs={}, response_metadata={}, id='0e36e82f-a8af-432a-b965-30a3b37a7986', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='what year he died?', additional_kwargs={}, response_metadata={}, id='00863330-58ab-47c7-9685-935d531ce38c'),
  AIMessage(content='Mahatma Gandhi passed away on January 30, 1948.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 105, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DUc0UxKPwhPwdMBdIuXohiQ4VKaOW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d8d12-c747-7ba0-b805-004cc575838e-0', t

However, there's an important detail to consider.

Note that the memory we've implemented so far only persists during the current execution.

In other words, if you rerun the graph, it won't remember anything from the previous conversation.

The memory resets with each new invocation.

Give it a try.

Rerun the compiled graph and watch the memory start fresh.

This is where thread level persistence comes in.

Retaining memory across multiple executions within the same conversation thread sounds mysterious and

intriguing.

